# 🕹️ Retro Arcade — Setup

> Build stamp: **2026-05-30 09:21:33**

Run this notebook **ONCE** to bootstrap the game. It populates the
`Arcade_LH` Lakehouse with synthetic 80s-arcade tables (Scores, Games,
Players, Cabinets, Date) and builds the `ArcadeHall_Model` Direct Lake
semantic model that you will use to build your report.

## Requirements
1. Attach **`Arcade_LH`** as the default Lakehouse (📚 icon → *Add* → Existing Lakehouse).
2. Run all cells top → bottom.

Total runtime: ~1–2 minutes.


In [ ]:
# === Setup ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0"],
               check=False, capture_output=True)

import random, datetime as dt
from pyspark.sql import functions as F, types as T

SEED_VERSION = "v1"
RNG_SEED = 1980
YEARS    = [2023, 2024, 2025]
random.seed(RNG_SEED)

for t in ["date", "games", "players", "cabinets", "scores",
          "Date", "Games", "Players", "Cabinets", "Scores"]:
    try: spark.sql(f"DROP TABLE IF EXISTS {t}")
    except Exception: pass
print("🧹 Clean slate.")


## Step 1 — Date dimension (3 years)


In [ ]:
start = dt.date(YEARS[0], 1, 1); end = dt.date(YEARS[-1], 12, 31)
days  = (end - start).days + 1
rows = []
for i in range(days):
    d = start + dt.timedelta(days=i)
    rows.append((int(d.strftime("%Y%m%d")), d, d.year,
                 (d.month - 1) // 3 + 1, d.month, d.strftime("%B"),
                 d.day, d.strftime("%A"), d.isoweekday() in (6, 7)))
schema = T.StructType([
    T.StructField("DateKey", T.IntegerType(), False),
    T.StructField("Date", T.DateType(), False),
    T.StructField("Year", T.IntegerType(), False),
    T.StructField("Quarter", T.IntegerType(), False),
    T.StructField("MonthNum", T.IntegerType(), False),
    T.StructField("MonthName", T.StringType(), False),
    T.StructField("DayOfMonth", T.IntegerType(), False),
    T.StructField("DayName", T.StringType(), False),
    T.StructField("IsWeekend", T.BooleanType(), False),
])
date_df = spark.createDataFrame(rows, schema)
(date_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable("date"))
print(f"✅ date rows={date_df.count():,}")


## Step 2 — Games dimension (12 arcade classics)


In [ ]:
GAMES = [
    (1,  "Pac-Man",        "Maze",     1980, "Namco"),
    (2,  "Donkey Kong",    "Platform", 1981, "Nintendo"),
    (3,  "Galaga",         "Shooter",  1981, "Namco"),
    (4,  "Frogger",        "Action",   1981, "Konami"),
    (5,  "Centipede",      "Shooter",  1981, "Atari"),
    (6,  "Asteroids",      "Shooter",  1979, "Atari"),
    (7,  "Q*bert",         "Puzzle",   1982, "Gottlieb"),
    (8,  "Defender",       "Shooter",  1981, "Williams"),
    (9,  "Dig Dug",        "Maze",     1982, "Namco"),
    (10, "Tempest",        "Shooter",  1981, "Atari"),
    (11, "Street Fighter II","Fighting",1991,"Capcom"),
    (12, "Tetris",         "Puzzle",   1984, "Atari Games"),
]
games_df = spark.createDataFrame(GAMES,
    ["GameKey", "GameName", "Genre", "ReleaseYear", "Manufacturer"])
(games_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable("games"))
print(f"✅ games rows={games_df.count():,}")


## Step 3 — Players dimension (40 handles)


In [ ]:
HANDLES = ["BLAZE","NEON","PIXEL","ROXY","ZARA","KIRO","MAXX","NOVA","JINX","RIPP",
           "VEGA","ECHO","FROST","HEX","ATOM","DUKE","SCAR","WOLF","BOLT","ASH",
           "ZERO","RAVN","FURY","NYX","TANK","KOBR","FLUX","JADE","ORC","KAI",
           "BANE","GHST","VYX","RUNE","ZED","PYRO","FAUN","ION","KARM","BYTE"]
COUNTRIES = ["US","JP","IT","FR","DE","UK","ES","BR","CA","KR"]
rng = random.Random(RNG_SEED + 1)
rows = [(i+1, h, rng.choice(COUNTRIES), rng.randint(1979, 1995))
        for i, h in enumerate(HANDLES)]
players_df = spark.createDataFrame(rows,
    ["PlayerKey", "Handle", "Country", "JoinYear"])
(players_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable("players"))
print(f"✅ players rows={players_df.count():,}")


## Step 4 — Cabinets dimension (8 arcade locations)


In [ ]:
CABINETS = [
    (1, "Times Square Hall",  "Upright"),
    (2, "Tokyo Akihabara",    "Upright"),
    (3, "Milano Galleria",    "Cocktail"),
    (4, "Paris Pigalle",      "Upright"),
    (5, "London Soho",        "Cocktail"),
    (6, "Berlin Friedrich",   "Upright"),
    (7, "LA Venice Beach",    "Cabaret"),
    (8, "Madrid Gran Via",    "Upright"),
]
cab_df = spark.createDataFrame(CABINETS, ["CabinetKey","Location","CabinetType"])
(cab_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable("cabinets"))
print(f"✅ cabinets rows={cab_df.count():,}")


## Step 5 — Scores fact (~80k rows, with seasonality & player skill)


In [ ]:
date_pool = [(int(d.strftime('%Y%m%d')), d) for d in
             (dt.date(YEARS[0], 1, 1) + dt.timedelta(days=i) for i in range(days))]
rng = random.Random(RNG_SEED + 2)

# Player skill 0.5..2.5 multiplier
skill = {pk: rng.uniform(0.5, 2.5) for pk in range(1, len(HANDLES)+1)}
# Game base score
base  = {gk: rng.randint(2_000, 15_000) for gk in range(1, len(GAMES)+1)}
# Weekend bonus
def factor(d, pk, gk):
    f = 1.0
    if d.isoweekday() in (6,7): f *= 1.25
    if d.month in (7,8,12):     f *= 1.15  # summer & xmas peak
    return f * skill[pk]

N_SCORES = 80_000
rows = []
for sk in range(1, N_SCORES + 1):
    dk, d  = rng.choice(date_pool)
    pk     = rng.randint(1, len(HANDLES))
    gk     = rng.randint(1, len(GAMES))
    cabk   = rng.randint(1, len(CABINETS))
    score  = int(base[gk] * rng.uniform(0.4, 1.6) * factor(d, pk, gk))
    dur    = rng.randint(30, 1800)              # seconds
    cred   = rng.randint(1, 5)
    onecc  = (cred == 1 and rng.random() < 0.05)  # 5% of single-credit runs are 1cc
    rows.append((sk, dk, pk, gk, cabk, score, dur, cred, onecc))

schema = T.StructType([
    T.StructField("ScoreKey", T.IntegerType(), False),
    T.StructField("DateKey",  T.IntegerType(), False),
    T.StructField("PlayerKey",T.IntegerType(), False),
    T.StructField("GameKey",  T.IntegerType(), False),
    T.StructField("CabinetKey",T.IntegerType(),False),
    T.StructField("Score",    T.IntegerType(), False),
    T.StructField("DurationSeconds", T.IntegerType(), False),
    T.StructField("Credits",  T.IntegerType(), False),
    T.StructField("OneCC",    T.BooleanType(), False),
])
scores_df = spark.createDataFrame(rows, schema)
(scores_df.write.format("delta").mode("overwrite")
          .option("overwriteSchema", "true").saveAsTable("scores"))
print(f"✅ scores rows={scores_df.count():,}")


## Step 6 — Summary


In [ ]:
for t in ["date","games","players","cabinets","scores"]:
    n = spark.table(t).count()
    print(f"  {t:<10} {n:>10,} rows")
print("\n🕹️  Lakehouse Arcade_LH is ready.")


## Step 7 — Build the `ArcadeHall_Model` Direct Lake semantic model

Creates the semantic model with relationships and 6 base measures so you can
immediately start building visuals in your report.


In [ ]:
import subprocess, sys, importlib
try:
    import sempy_labs as labs
    from sempy_labs import directlake as labs_dl
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link-labs"],
                   check=True)
    importlib.invalidate_caches()
    import sempy_labs as labs
    from sempy_labs import directlake as labs_dl
print(f"sempy-labs version: {getattr(labs, '__version__', '?')}")

import sempy.fabric as fabric

MODEL_NAME = "ArcadeHall_Model"
LAKEHOUSE  = "Arcade_LH"
TABLES = {
    "Date":     "date",
    "Games":    "games",
    "Players":  "players",
    "Cabinets": "cabinets",
    "Scores":   "scores",
}

try:
    df = fabric.list_datasets()
    name_col = next((c for c in df.columns
                     if c.lower() in ("dataset name","name","display name")), None)
    already = (df[name_col] == MODEL_NAME).any() if name_col else False
except Exception as e:
    print(f"(could not list datasets: {e}); will try to create.")
    already = False

if already:
    print(f"✅ Semantic model '{MODEL_NAME}' already exists — refreshing.")
    try:
        fabric.refresh_dataset(dataset=MODEL_NAME, refresh_type="full")
        print("✅ Refresh OK.")
    except Exception as e:
        print(f"⚠️  Refresh failed: {e}")
else:
    import time
    print("⏳ Waiting 45s for SQL endpoint metadata sync of newly-written tables...")
    time.sleep(45)

    print(f"🛠️  Creating Direct Lake model '{MODEL_NAME}' from Lakehouse '{LAKEHOUSE}' (no refresh)...")
    labs_dl.generate_direct_lake_semantic_model(
        dataset=MODEL_NAME,
        tables=TABLES,
        source=LAKEHOUSE,
        source_type="Lakehouse",
        refresh=False,
    )
    print(f"✅ Created '{MODEL_NAME}' (unrefreshed).")

    # First refresh on a brand-new Direct Lake model can flake while the
    # SQL endpoint catches up — retry a few times.
    print("⏳ Refreshing model (with retries)...")
    last_err = None
    for attempt in range(1, 6):
        try:
            fabric.refresh_dataset(dataset=MODEL_NAME, refresh_type="full")
            print(f"✅ Refresh succeeded on attempt {attempt}.")
            last_err = None
            break
        except Exception as e:
            last_err = e
            print(f"   attempt {attempt} failed: {e}")
            time.sleep(20)
    if last_err is not None:
        print("⚠️  Refresh still failing. The model exists, but Step 8 may need to wait — re-run this cell in a minute.")
        print(f"   Last error: {last_err}")


## Step 8 — Add relationships and base measures


In [ ]:
from sempy_labs.tom import connect_semantic_model

with connect_semantic_model(dataset=MODEL_NAME, readonly=False) as tom:
    # --- Rename physical tables to PascalCase ---
    wanted = {"date":"Date","games":"Games","players":"Players",
              "cabinets":"Cabinets","scores":"Scores"}
    for t in list(tom.model.Tables):
        if t.Name in wanted and t.Name != wanted[t.Name]:
            t.Name = wanted[t.Name]

    # --- Mark Date table ---
    try:
        tom.mark_as_date_table(table_name="Date", column_name="Date")
    except Exception as e:
        print(f"(mark_as_date_table skipped: {e})")

    # --- Relationships (delete pre-existing autos, then create explicit) ---
    for r in list(tom.model.Relationships):
        tom.model.Relationships.Remove(r)

    rels = [
        ("Scores","DateKey","Date","DateKey"),
        ("Scores","GameKey","Games","GameKey"),
        ("Scores","PlayerKey","Players","PlayerKey"),
        ("Scores","CabinetKey","Cabinets","CabinetKey"),
    ]
    for ft, fk, dt, dk in rels:
        try:
            tom.add_relationship(
                from_table=ft, from_column=fk,
                to_table=dt,    to_column=dk,
                from_cardinality="Many", to_cardinality="One",
            )
        except Exception as e:
            print(f"(relationship {ft}->{dt} skipped: {e})")

    # --- Base measures on Scores ---
    measures = [
        ("Total Score",       "SUM(Scores[Score])",               "#,0"),
        ("Total Plays",       "COUNTROWS(Scores)",                "#,0"),
        ("Total Credits",     "SUM(Scores[Credits])",             "#,0"),
        ("Active Players",    "DISTINCTCOUNT(Scores[PlayerKey])", "0"),
        ("Cabinets Used",     "DISTINCTCOUNT(Scores[CabinetKey])","0"),
        ("Avg Score",         "AVERAGE(Scores[Score])",           "#,0"),
        ("1cc Achievements",  "CALCULATE(COUNTROWS(Scores), Scores[OneCC] = TRUE)", "#,0"),
    ]
    existing = {m.Name for m in tom.all_measures()}
    for name, expr, fmt in measures:
        if name in existing: continue
        tom.add_measure(table_name="Scores", measure_name=name,
                        expression=expr, format_string=fmt,
                        description="Base measure provided by setup.")
    print("✅ Relationships + measures applied.")

# Renames + structural changes in Direct Lake invalidate the cached partition.
# Force a refresh now so the report can query the model immediately.
print("⏳ Refreshing model after rename / relationship / measure changes...")
import time as _time
_last = None
for attempt in range(1, 6):
    try:
        fabric.refresh_dataset(dataset=MODEL_NAME, refresh_type="full")
        print(f"✅ Post-edit refresh OK (attempt {attempt}).")
        _last = None
        break
    except Exception as e:
        _last = e
        print(f"   attempt {attempt} failed: {e}")
        _time.sleep(20)
if _last is not None:
    print("⚠️  Refresh still failing. Wait a minute and re-run THIS cell, or refresh the model manually from the workspace.")
    print(f"   Last error: {_last}")


## ✅ Done

`ArcadeHall_Model` is live. Next:
1. Open **`02_Quest`** for the 5-level brief.
2. In the workspace, **+ New → Power BI report → Pick a published semantic model →
   `ArcadeHall_Model`** and name the report **`Arcade_Hall_Report`**.
3. Build the report following the 5 levels.
4. When done, open **`03_Check`** to validate and mint your badge.
